# Исследование надежности заемщиков


In [1]:
import pandas as pd

try:
    data = pd.read_csv('/datasets/data.csv')
except:
    data = pd.read_csv('https://code.s3.yandex.net/datasets/data.csv')

In [2]:
data.head(20)

,children,days_employed,dob_years,education,education_id,family_status,family_status_id,gender,income_type,debt,total_income,purpose
0,1,-8437.673028,42,высшее,0,женат / замужем,0,F,сотрудник,0,253875.639453,покупка жилья
1,1,-4024.803754,36,среднее,1,женат / замужем,0,F,сотрудник,0,112080.014102,приобретение автомобиля
2,0,-5623.422610,33,Среднее,1,женат / замужем,0,M,сотрудник,0,145885.952297,покупка жилья
3,3,-4124.747207,32,среднее,1,женат / замужем,0,M,сотрудник,0,267628.550329,дополнительное образование
4,0,340266.072047,53,среднее,1,гражданский брак,1,F,пенсионер,0,158616.077870,сыграть свадьбу
5,0,-926.185831,27,высшее,0,гражданский брак,1,M,компаньон,0,255763.565419,покупка жилья
6,0,-2879.202052,43,высшее,0,женат / замужем,0,F,компаньон,0,240525.971920,операции с жильем
7,0,-152.779569,50,СРЕДНЕЕ,1,женат / замужем,0,M,сотрудник,0,135823.934197,образование
8,2,-6929.865299,35,ВЫСШЕЕ,0,гражданский брак,1,F,сотрудник,0,95856.832424,на проведение свадьбы
9,0,-2188.756445,41,среднее,1,женат / замужем,0,M,сотрудник,0,144425.938277,покупка жилья для семьи


In [3]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21525 entries, 0 to 21524
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   children          21525 non-null  int64  
 1   days_employed     19351 non-null  float64
 2   dob_years         21525 non-null  int64  
 3   education         21525 non-null  object 
 4   education_id      21525 non-null  int64  
 5   family_status     21525 non-null  object 
 6   family_status_id  21525 non-null  int64  
 7   gender            21525 non-null  object 
 8   income_type       21525 non-null  object 
 9   debt              21525 non-null  int64  
 10  total_income      19351 non-null  float64
 11  purpose           21525 non-null  object 
dtypes: float64(2), int64(5), object(5)
memory usage: 2.0+ MB


## Предобработка данных

### Удаление пропусков

In [4]:
data.isna().sum()

children               0
days_employed       2174
dob_years              0
education              0
education_id           0
family_status          0
family_status_id       0
gender                 0
income_type            0
debt                   0
total_income        2174
purpose                0
dtype: int64

In [5]:
for t in data['income_type'].unique():
    data.loc[(data['income_type'] == t) & (data['total_income'].isna()), 'total_income'] = \
    data.loc[(data['income_type'] == t), 'total_income'].median()

### Обработка аномальных значений

In [6]:
data['days_employed'] = data['days_employed'].abs()

In [7]:
data.groupby('income_type')['days_employed'].agg('median')

income_type
безработный        366413.652744
в декрете            3296.759962
госслужащий          2689.368353
компаньон            1547.382223
пенсионер          365213.306266
предприниматель       520.848083
сотрудник            1574.202821
студент               578.751554
Name: days_employed, dtype: float64

У двух типов (безработные и пенсионеры) получатся аномально большие значения. Исправить такие значения сложно, без дополнительных исследований, в рамках данного исследования оставим их без изменений.

In [8]:
data['children'].unique()

array([ 1,  0,  3,  2, -1,  4, 20,  5])

In [9]:
data = data[(data['children'] != -1) & (data['children'] != 20)]

In [10]:
data['children'].unique()

array([1, 0, 3, 2, 4, 5])

### Удаление пропусков (продолжение)

In [11]:
for t in data['income_type'].unique():
    data.loc[(data['income_type'] == t) & (data['days_employed'].isna()), 'days_employed'] = \
    data.loc[(data['income_type'] == t), 'days_employed'].median()

In [12]:
data.isna().sum()

children            0
days_employed       0
dob_years           0
education           0
education_id        0
family_status       0
family_status_id    0
gender              0
income_type         0
debt                0
total_income        0
purpose             0
dtype: int64

### Изменение типов данных

In [13]:
data['total_income'] = data['total_income'].astype(int)

### Обработка дубликатов

In [14]:
data['education'] = data['education'].str.lower()

In [15]:
data.duplicated().sum()

71

In [16]:
data = data.drop_duplicates()

### Категоризация данных

In [17]:
def categorize_income(income):
    try:
        if 0 <= income <= 30000:
            return 'E'
        elif 30001 <= income <= 50000:
            return 'D'
        elif 50001 <= income <= 200000:
            return 'C'
        elif 200001 <= income <= 1000000:
            return 'B'
        elif income >= 1000001:
            return 'A'
    except:
        pass

In [18]:
data['total_income_category'] = data['total_income'].apply(categorize_income)

In [19]:
data['purpose'].unique()

array(['покупка жилья', 'приобретение автомобиля',
       'дополнительное образование', 'сыграть свадьбу',
       'операции с жильем', 'образование', 'на проведение свадьбы',
       'покупка жилья для семьи', 'покупка недвижимости',
       'покупка коммерческой недвижимости', 'покупка жилой недвижимости',
       'строительство собственной недвижимости', 'недвижимость',
       'строительство недвижимости', 'на покупку подержанного автомобиля',
       'на покупку своего автомобиля',
       'операции с коммерческой недвижимостью',
       'строительство жилой недвижимости', 'жилье',
       'операции со своей недвижимостью', 'автомобили',
       'заняться образованием', 'сделка с подержанным автомобилем',
       'получение образования', 'автомобиль', 'свадьба',
       'получение дополнительного образования', 'покупка своего жилья',
       'операции с недвижимостью', 'получение высшего образования',
       'свой автомобиль', 'сделка с автомобилем',
       'профильное образование', 'высшее об

In [20]:
def categorize_purpose(row):
    try:
        if 'автом' in row:
            return 'операции с автомобилем'
        elif 'жил' in row or 'недвиж' in row:
            return 'операции с недвижимостью'
        elif 'свад' in row:
            return 'проведение свадьбы'
        elif 'образов' in row:
            return 'получение образования'
    except:
        return 'нет категории'

In [21]:
data['purpose_category'] = data['purpose'].apply(categorize_purpose)

#### Есть ли зависимость между количеством детей и возвратом кредита в срок?

Для того чтобы ответить на этот вопрос создадим сводную таблицу, в которую внесем данные о количестве детей у клиентов и наличие у них задолженности. Также, создадим отдельный столбец, в котором подсчитаем долю клиентов с задолженностями.

In [22]:
children_data_pivot = data.pivot_table(index=['children'], columns='debt', values='purpose_category', aggfunc='count')

# заполним нулевое значение в сводой таблице
children_data_pivot =  children_data_pivot.fillna(0) 

# расчитаем долю невозврата кредита у клиентов с одинаковым количеством детей
children_data_pivot['percent'] = (children_data_pivot[1] / (children_data_pivot[1] + children_data_pivot[0])) * 100 

# переведем данные в цельночисленный формат
children_data_pivot[1] = children_data_pivot[1].astype('int')     
children_data_pivot[0] = children_data_pivot[0].astype('int')

children_data_pivot

debt,0,1,percent
children,,,
0,13028,1063,7.543822
1,4364,444,9.234609
2,1858,194,9.454191
3,303,27,8.181818
4,37,4,9.756098
5,9,0,0.000000


In [23]:
# переименуем столбцы для простоты интерпретации
children_data_pivot.rename(columns={0: 'нет задолженности', 1: 'есть задолженность', 'percent': 'доля должников'}, inplace=True)

children_data_pivot.sort_values('доля должников')

debt,нет задолженности,есть задолженность,доля должников
children,,,
5,9,0,0.000000
0,13028,1063,7.543822
3,303,27,8.181818
1,4364,444,9.234609
2,1858,194,9.454191
4,37,4,9.756098


**Вывод:** Самая высокая доля должников по кредиту у клиентов с 4мя детьми. Однако, стоит отметить, что общее число клиентов с 4мя детьми всего 41 человек, тогда как, например, у клиентов с 2мя детьми доля должников имеет близкие показатели, но таких клиентов в банке уже 2025. Совсем нет задолженностей у клиентов с 5ью детьми, но таких клиентов в банке всего 9 человек, что не дает нам возможности полноценно оценивать этот показатель (ситуация с 5 детьми в семье довольно редкая в современных реалиях). Опираясь на использованные данные, мы можем сказать, что количество детей влияет на возврат кредита в срок. Наименьший процент должников у бездетных клиентов.

#### Есть ли зависимость между семейным положением и возвратом кредита в срок?

Для ответа на данный вопрос в создании сводной таблицы используем данные столбца **family_status**.

In [24]:
family_data_pivot = data.pivot_table(index=['family_status'], columns='debt', values='purpose_category', aggfunc='count')

# расчитаем долю невозврата кредита у клиентов с одинаковым семейным положением
family_data_pivot['percent'] = family_data_pivot[1] / (family_data_pivot[1] + family_data_pivot[0]) * 100

# переведем данные в цельночисленный формат
family_data_pivot[0] = family_data_pivot[0].astype('int')
family_data_pivot[1] = family_data_pivot[1].astype('int')
family_data_pivot.sort_values('percent')

debt,0,1,percent
family_status,,,
вдовец / вдова,888,63,6.624606
в разводе,1105,84,7.064760
женат / замужем,11334,927,7.560558
гражданский брак,3749,385,9.313014
Не женат / не замужем,2523,273,9.763948


In [25]:
# переименуем столбцы для простоты интерпретации
family_data_pivot.rename(columns={0: 'нет задолженности', 1: 'есть задолженность', 'percent': 'доля должников'}, inplace=True)
family_data_pivot.sort_values('доля должников')

debt,нет задолженности,есть задолженность,доля должников
family_status,,,
вдовец / вдова,888,63,6.624606
в разводе,1105,84,7.064760
женат / замужем,11334,927,7.560558
гражданский брак,3749,385,9.313014
Не женат / не замужем,2523,273,9.763948


**Вывод:** Самая высокая доля должников у клиентов не состоящих в браке. Чуть ниже показатели у клиентов, состоящих в гражданском браке. Процент должников ниже у категорий клиентов, состоящих когда-либо в браке(в настоящем или прошлом). 

#### Есть ли зависимость между уровнем дохода и возвратом кредита в срок?

Для ответа на данный вопрос используем ранее созданный нами столбец **total_income_category**. 

In [26]:
income_data_pivot = data.pivot_table(index=['total_income_category'], columns='debt', values='gender', aggfunc='count')

# расчитаем долю невозврата кредита у клиентов с одинаковым уровнем дохода
income_data_pivot['percent'] = income_data_pivot[1] / (income_data_pivot[1] + income_data_pivot[0]) * 100

# переведем данные в цельночисленный формат
income_data_pivot[0] = income_data_pivot[0].astype('int')
income_data_pivot[1] = income_data_pivot[1].astype('int')
income_data_pivot.sort_values('percent')

debt,0,1,percent
total_income_category,,,
D,328,21,6.017192
B,4660,354,7.060231
A,23,2,8.000000
C,14568,1353,8.498210
E,20,2,9.090909


In [27]:
# переименуем столбцы для простоты интерпретации
income_data_pivot.rename(columns={0: 'нет задолженности', 1: 'есть задолженность', 'percent': 'доля должников'}, inplace=True)
income_data_pivot.sort_values('доля должников')

debt,нет задолженности,есть задолженность,доля должников
total_income_category,,,
D,328,21,6.017192
B,4660,354,7.060231
A,23,2,8.000000
C,14568,1353,8.498210
E,20,2,9.090909


**Вывод:** Вспомним, как соотносятся буквенные категории с уровнем дохода клиентов: 
0–30000 — 'E';
30001–50000 — 'D';
50001–200000 — 'C';
200001–1000000 — 'B';
1000001 и выше — 'A'.
Стоит отметить, что клиенты распределились неравномерно, клиенты с категорией дохода B и C самые многочисленные. Самыми ответственными клиентами оказались клиенты с категорией дохода D. А самый большой показатель задолженности оказался у клиентов с наименьшим доходом. Однако, стоит учесть, что их группа довольно малочисленна, поэтому отметим также задолжников из категории дохода C.

#### Как разные цели кредита влияют на его возврат в срок?

Для ответа на данный вопрос применим данные из созданного нами ранее столбца **purpose_category**.

In [28]:
purpose_data_pivot = data.pivot_table(index=['purpose_category'], columns='debt', values='gender', aggfunc='count')

# расчитаем долю невозврата кредита у клиентов с одинаковыми кредитными целями
purpose_data_pivot['percent'] = purpose_data_pivot[1] / (purpose_data_pivot[1] + purpose_data_pivot[0]) * 100

# переведем данные в цельночисленный формат
purpose_data_pivot[0] = purpose_data_pivot[0].astype('int')
purpose_data_pivot[1] = purpose_data_pivot[1].astype('int')
purpose_data_pivot.sort_values('percent')

debt,0,1,percent
purpose_category,,,
операции с недвижимостью,9971,780,7.255139
проведение свадьбы,2130,183,7.911803
получение образования,3619,369,9.252758
операции с автомобилем,3879,400,9.347978


In [29]:
# переименуем столбцы для простоты интерпретации
purpose_data_pivot.rename(columns={0: 'нет задолженности', 1: 'есть задолженность', 'percent': 'доля должников'}, inplace=True)
purpose_data_pivot.sort_values('доля должников')

debt,нет задолженности,есть задолженность,доля должников
purpose_category,,,
операции с недвижимостью,9971,780,7.255139
проведение свадьбы,2130,183,7.911803
получение образования,3619,369,9.252758
операции с автомобилем,3879,400,9.347978


**Вывод:** Наименьшая доля должников у клиентов совершающих операции с недвижимостью, наибольшая доля должников у клиентов, которые берут кредит на операции с автомобилем. 

#### Приведем возможные причины появления пропусков в исходных данных.

Пропуски данных могут возникнуть по нескольким причинам. Можем предположить, что при заполнении анкет клиентом некоторые могли упустить определенные графы для заполнения. Также, возможно сотрудники банка могли допустить ошибку при внесении данных о клиенте. Отметим также возможность случайного или намеренного некорректного удаления данных из таблицы (например, чтобы убрать дубликаты из таблицы). Исходя из этого особенно важно подготовить данные к последующему исследованию для более точной проверки гипотез.

#### Почему заполнить пропуски медианным значением подходящее решение в данном исследовании?

Медиана - центральное значение упорядоченного набора данных. Если заполнять пропуски средним значением, на него заметно влияют крайние экстримальные значения, в случае с медианой таких искажений можно избежать. Медиана позволяет сохранить естественное распределение, что делает анализ данных более точным.

### Общий вывод

В результате исследования было выявленно, что наиболее ответственными клиентами банка оказались бездетные клиенты, клиенты, состоящие(или состоявшие) в браке. У ответственных заемщиков уровень дохода в диапозоне от 30001 до 50000. Категория, в которой клиенты чаще возвращают кредит в срок - это операции с недвижимостью. 
Банку стоит тщательнее обрабатывать заявки от одиноких клиентов, с 2мя и более детьми, так как у этих категорий заемщиков наибольший процент должников по кредитам. Также, особое внимание следует уделить заемщикам, которые берут кредит на операции с автомобилем и имеют низкий средний доход. 